# QLoRA Fine-Tuning: Qwen2.5-Coder-7B-Instruct for SVA Generation

Fine-tunes Qwen2.5-Coder-7B-Instruct using QLoRA to generate SystemVerilog Assertions (SVA) from RTL.

**Ablation Configs** — change `DATASET_CONFIG` below:
- `all` — unfiltered (includes non-compiling)
- `syntax_pass` — compiles in Jasper
- `verified` — compiles AND passes verification

**Requirements**: Colab Pro with A100 GPU.

## 0. Install Dependencies

In [1]:
!pip install -q \
    torch \
    transformers>=4.44.0 \
    datasets \
    accelerate \
    peft>=0.12.0 \
    trl>=0.9.0 \
    bitsandbytes>=0.43.0 \
    wandb \
    huggingface_hub

## 1. Configuration

**Change `DATASET_CONFIG` for each ablation run.**

In [ ]:
# ============================================================
# CHANGE THESE BETWEEN ABLATION RUNS
# ============================================================
DATASET_CONFIG = "all"  # Options: "all", "syntax_pass", "verified"
RUN_NAME = f"sva-qlora-{DATASET_CONFIG}"

# ============================================================
# Dataset
# ============================================================
DATASET_REPO = "aarushgoradia/malik25_26"

# ============================================================
# Model
# ============================================================
BASE_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"

# ============================================================
# QLoRA Hyperparameters
# ============================================================
LORA_RANK = 64
LORA_ALPHA = 128          # 2x rank is standard
LORA_DROPOUT = 0.05
TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# ============================================================
# Training Hyperparameters
# ============================================================
NUM_EPOCHS = 3
BATCH_SIZE = 2              # per-device
GRADIENT_ACCUMULATION = 8   # effective batch = 2 * 8 = 16
LEARNING_RATE = 2e-4
LR_SCHEDULER = "cosine"
WARMUP_RATIO = 0.05
MAX_SEQ_LENGTH = 8192
WEIGHT_DECAY = 0.01

# ============================================================
# Output
# ============================================================
OUTPUT_DIR = f"/content/drive/MyDrive/sva_qlora/{RUN_NAME}"
PUSH_TO_HUB = False        # set True to push adapter to HF
HF_ADAPTER_REPO = f"aarushgoradia/sva-qlora-{DATASET_CONFIG}"

In [3]:
# FOR VERT ONLY
# ============================================================
# VERT Dataset Run
# ============================================================
RUN_NAME = "sva-qlora-vert"

# ============================================================
# Dataset
# ============================================================
VERT_JSON_PATH = "VERT.json"  # update path if needed

# ============================================================
# Model
# ============================================================
BASE_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"

# ============================================================
# QLoRA Hyperparameters
# ============================================================
LORA_RANK = 64
LORA_ALPHA = 128          # 2x rank is standard
LORA_DROPOUT = 0.05
TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# ============================================================
# Training Hyperparameters
# ============================================================
NUM_EPOCHS = 1
BATCH_SIZE = 2              # per-device
GRADIENT_ACCUMULATION = 8   # effective batch = 2 * 8 = 16
LEARNING_RATE = 2e-4
LR_SCHEDULER = "cosine"
WARMUP_RATIO = 0.05
MAX_SEQ_LENGTH = 2048       # consider dropping to 2048 after length check below
WEIGHT_DECAY = 0.01

# ============================================================
# Output
# ============================================================
OUTPUT_DIR = f"/content/drive/MyDrive/sva_qlora/{RUN_NAME}"
PUSH_TO_HUB = True        # set True to push adapter to HF
HF_ADAPTER_REPO = "aarushgoradia/sva-qlora-vert"

## 2. Login & Setup

In [4]:
from huggingface_hub import notebook_login
notebook_login()

In [5]:
# Optional: Weights & Biases logging
import wandb
wandb.login()
wandb.init(project="sva-qlora-ablation", entity="aarush-goradia-princeton-university", name=RUN_NAME)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: aarush-goradia (aarush-goradia-princeton-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [6]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Verify GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")

Mounted at /content/drive
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


## 3. Load & Format Dataset

In [6]:
from datasets import load_dataset

dataset = load_dataset(DATASET_REPO, DATASET_CONFIG)
print(f"Config: {DATASET_CONFIG}")
print(f"Train: {len(dataset['train'])} | Val: {len(dataset['validation'])} | Test: {len(dataset['test'])}")

NameError: name 'DATASET_REPO' is not defined

In [7]:
# ============================================================
# System prompt — same one used for ChatGPT generation
# ============================================================
SYSTEM_PROMPT = """\
You are an expert SystemVerilog verification engineer writing SVA for Jasper \
formal verification.
OUTPUT REQUIREMENTS:
1. Output a single, complete .sv file that compiles in Jasper without modification.
2. The file must be a module that takes the DUT's ports as inputs and contains \
   SVA properties bound to those signals. You can use internal signals \
   if they are present in the RTL, but do NOT invent new ones.
3. Every property must use a clocked event (@(posedge clk) or the appropriate \
   clock from the RTL). NEVER use combinational or level-sensitive events in \
   property statements — Jasper rejects these. For modules with combinational \
   logic, still clock your assertions to the appropriate clock edge.
4. Use `disable iff` with the correct reset polarity as shown in the RTL.
5. Use descriptive labels for every assertion (e.g., `check_grant_mutex`, not `a1`).
6. Add a brief comment above each assertion explaining what it checks.
7. Only assert behaviors that the RTL actually implements. Do not invent \
   signals, states, or protocols that are not present in the code.
8. Focus on QUALITY and CORRECTNESS — 10 correct, meaningful assertions are \
   worth more than 30 trivial or speculative ones.
9. Do NOT wrap output in markdown code fences or add explanation outside the code.
10. Keep comments minimal — one short line per assertion. Do NOT include large \
   comment blocks, file headers, or explanations of your approach.
REFERENCE EXAMPLE — this is the style and quality level to target:
```systemverilog
module manual (
    input logic CLK,
    input logic RESETn,
    input logic QREQn,
    input logic QACCEPTn,
    input logic QDENY,
    input logic QACTIVE
);
    ///// Handshake rules /////
    // QREQn can only transition from HIGH to LOW when QACCEPTn is HIGH and QDENY is LOW.
    handshake_1: assume property (
        @(posedge CLK) disable iff (!RESETn) $fell(QREQn) |-> (QACCEPTn == 1'b1) && (QDENY == 1'b0)
    );
    // QACCEPTn can only transition from HIGH to LOW when QREQn is LOW and QDENY is LOW.
    handshake_3: assert property (
        @(posedge CLK) disable iff (!RESETn) $fell(QACCEPTn) |-> (QREQn == 1'b0) && (QDENY == 1'b0)
    );
    // QDENY can only transition from LOW to HIGH when QREQn is LOW and QACCEPTn is HIGH.
    handshake_6: assert property (
        @(posedge CLK) disable iff (!RESETn) $rose(QDENY) |-> (QREQn == 1'b0) && (QACCEPTn == 1'b1)
    );
    ///// Device reset /////
    // At reset assertion, a device must drive both QACCEPTn and QDENY LOW.
    reset: assert property (
        @(posedge CLK) !RESETn |-> (QACCEPTn == 1'b0) && (QDENY == 1'b0)
    );
endmodule
```
Note the pattern: module wrapper with DUT ports as inputs, descriptive labels, \
comments explaining intent, proper clocking and reset disable on every property, \
and appropriate use of assume vs assert."""


def make_user_prompt(rtl_code: str) -> str:
    return f"""Analyze the following RTL module carefully. Identify:
- The clock(s) and reset signal(s), including reset polarity
- Whether the logic is sequential, combinational, or mixed
- The key signals, interfaces, and functional behaviors
Then generate a complete .sv assertion module following the style shown in \
your reference example. Only write assertions for behaviors that are actually \
present in this RTL — do not guess or assume functionality that isn't there. \
For combinational logic, still use clocked assertions (@(posedge clk)).
RTL module:
```verilog
{rtl_code}
```"""

In [ ]:
def format_chat(example):
    """Convert a dataset row into Qwen chat format."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": make_user_prompt(example["rtl"])},
        {"role": "assistant", "content": example["sva"]},
    ]
    return {"messages": messages}

train_dataset = dataset["train"].map(format_chat, remove_columns=dataset["train"].column_names)
val_dataset = dataset["validation"].map(format_chat, remove_columns=dataset["validation"].column_names)

print(f"Formatted — Train: {len(train_dataset)} | Val: {len(val_dataset)}")
print(f"\nSample messages[0] role: {train_dataset[0]['messages'][0]['role']}")
print(f"Sample messages[1] role: {train_dataset[0]['messages'][1]['role']}")
print(f"Sample messages[2] role: {train_dataset[0]['messages'][2]['role']}")

Map:   0%|          | 0/4619 [00:00<?, ? examples/s]

Map:   0%|          | 0/577 [00:00<?, ? examples/s]

Formatted — Train: 4619 | Val: 577

Sample messages[0] role: system
Sample messages[1] role: user
Sample messages[2] role: assistant


In [9]:
# FOR VERT ONLY

import json
from datasets import Dataset
from sklearn.model_selection import train_test_split

# Load JSONL
with open(VERT_JSON_PATH, "r") as f:
    raw = [json.loads(line) for line in f if line.strip()]

print(f"Total examples: {len(raw)}")

# Filter to synchronous only — combinational assertions conflict with Jasper system prompt
raw_sync = [ex for ex in raw if ex["Synchronous"] == "True"]
print(f"Synchronous only: {len(raw_sync)}")

# 90/5/5 split
train_raw, temp  = train_test_split(raw_sync, test_size=0.1, random_state=42)
val_raw, test_raw = train_test_split(temp,     test_size=0.5, random_state=42)

def format_chat_vert(example):
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": make_user_prompt(example["Code"])},
        {"role": "assistant", "content": example["Assertion"]},
    ]
    return {"messages": messages}

train_dataset = Dataset.from_list(train_raw).map(format_chat_vert, remove_columns=["Code", "Assertion", "Synchronous", "Clock"])
val_dataset   = Dataset.from_list(val_raw).map(format_chat_vert,   remove_columns=["Code", "Assertion", "Synchronous", "Clock"])

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_raw)}")

Total examples: 20000
Synchronous only: 10000


Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Train: 9000 | Val: 500 | Test: 500


## 4. Load Model (4-bit Quantized)

In [10]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)
model.config.use_cache = False  # required for gradient checkpointing

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.padding_side = "right"

# Qwen uses <|endoftext|> as pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded: {BASE_MODEL}")
print(f"Pad token: {tokenizer.pad_token} (id={tokenizer.pad_token_id})")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Model loaded: Qwen/Qwen2.5-Coder-7B-Instruct
Pad token: <|endoftext|> (id=151643)


## 5. Configure LoRA

In [11]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 161,480,704 || all params: 7,777,097,216 || trainable%: 2.0764


In [16]:
lengths = [
    len(tokenizer.encode(
        tokenizer.apply_chat_template(ex["messages"], tokenize=False, add_generation_prompt=False)
    ))
    for ex in train_dataset.select(range(200))
]
print(f"p50={sorted(lengths)[100]} p95={sorted(lengths)[190]} max={max(lengths)}")

p50=1288 p95=1713 max=1795


In [14]:
print(train_dataset[0])
print(train_dataset[0]["messages"])

{'messages': [{'role': 'system', 'content': "You are an expert SystemVerilog verification engineer writing SVA for Jasper formal verification.\nOUTPUT REQUIREMENTS:\n1. Output a single, complete .sv file that compiles in Jasper without modification.\n2. The file must be a module that takes the DUT's ports as inputs and contains    SVA properties bound to those signals. You can use internal signals    if they are present in the RTL, but do NOT invent new ones.\n3. Every property must use a clocked event (@(posedge clk) or the appropriate    clock from the RTL). NEVER use combinational or level-sensitive events in    property statements — Jasper rejects these. For modules with combinational    logic, still clock your assertions to the appropriate clock edge.\n4. Use `disable iff` with the correct reset polarity as shown in the RTL.\n5. Use descriptive labels for every assertion (e.g., `check_grant_mutex`, not `a1`).\n6. Add a brief comment above each assertion explaining what it checks.\

## 6. Train

In [12]:
import trl
print(trl.__version__)

1.4.0


In [14]:
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    run_name=RUN_NAME,

    # Epochs & batching
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,

    # Optimizer
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    optim="paged_adamw_8bit",

    # Sequence
    max_length=MAX_SEQ_LENGTH,

    # Precision
    bf16=True,
    fp16=False,

    # Evaluation & saving
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Logging
    logging_steps=10,
    report_to="wandb",

    # Memory
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    # Train on completions only — masks prompt tokens, only backprops through assistant response
    completion_only_loss=True,

    # Hub
    push_to_hub=PUSH_TO_HUB,
    hub_model_id=HF_ADAPTER_REPO if PUSH_TO_HUB else None,

    # Dataset
    dataset_kwargs={"skip_prepare_dataset": False},
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

print(f"Training {RUN_NAME}...")
print(f"  Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION}")
print(f"  Total steps: ~{len(train_dataset) * NUM_EPOCHS // (BATCH_SIZE * GRADIENT_ACCUMULATION)}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Tokenizing train dataset:   0%|          | 0/9000 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Training sva-qlora-vert...
  Effective batch size: 16
  Total steps: ~562


In [15]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
50,0.163906,0.164487
100,0.155523,0.155564
150,0.154039,0.153569
200,0.154035,0.151627
250,0.151098,0.151520
300,0.147751,0.148610
350,0.147570,0.146828
400,0.144527,0.145359
450,0.147355,0.143790
500,0.143946,0.142738


TrainOutput(global_step=563, training_loss=0.18420724864539625, metrics={'train_runtime': 5446.3897, 'train_samples_per_second': 1.652, 'train_steps_per_second': 0.103, 'total_flos': 5.679679852497101e+17, 'train_loss': 0.18420724864539625})

## 7. Save Adapter

In [16]:
PUSH_TO_HUB = True

In [17]:
# Save the best adapter
final_path = os.path.join(OUTPUT_DIR, "final_adapter")
trainer.save_model(final_path)
tokenizer.save_pretrained(final_path)
print(f"Adapter saved to: {final_path}")

# Optionally push to Hub
if PUSH_TO_HUB:
    trainer.push_to_hub()
    print(f"Pushed to: {HF_ADAPTER_REPO}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ra-vert/training_args.bin: 100%|##########| 5.78kB / 5.78kB            

  ...adapter/training_args.bin: 100%|##########| 5.78kB / 5.78kB            

  ...al_adapter/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

...

  ...al_adapter/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

  ...adapter_model.safetensors:  37%|###7      |  240MB /  646MB            

  ...adapter_model.safetensors:  37%|###7      |  240MB /  646MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Pushed to: aarushgoradia/sva-qlora-vert


## 8. Quick Sanity Check

Generate SVA for one test sample to make sure the adapter is working.

In [18]:
from peft import PeftModel

# Reload for inference (if needed after training)
# model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb_config, device_map="auto", torch_dtype=torch.bfloat16, trust_remote_code=True)
# model = PeftModel.from_pretrained(model, final_path)

model.eval()

# Grab one test example
import random
test_example_raw = random.choice(test_raw)  # test_raw is already in memory from Section 3
test_example = {"rtl": test_example_raw["Code"], "sva": test_example_raw["Assertion"]}

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": make_user_prompt(test_example["rtl"])},
]

text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=2048,
        temperature=0.1,
        top_p=0.95,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
    )

generated = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("=" * 80)
print("GENERATED SVA:")
print("=" * 80)
print(generated)

GENERATED SVA:
property ClockSynceotid; @(negedge async_clk_12) (  auth_1  && hw_6 ) |-> sig_10 == fsm_5 ; endproperty 
 
 property AuthCheckeotid; @(negedge async_clk_12) (  auth_1  && hw_6 ) && (  chip_7  || chip_3  && auth_12 ) |-> clk_14 == tx_7 ; endproperty 
 
 property ValidTxeotid; @(negedge async_clk_12) (  auth_1  && hw_6 ) &&  ( !(  chip_7  || chip_3  && auth_12 ) && (  rst_99 ) )  |-> err_14 == sig_6 ; endproperty 
 
 property ResetOnAutheotid; @(negedge async_clk_12) (  auth_1  && hw_6 ) && (  !(  chip_7  || chip_3  && auth_12 )  && !(  rst_99 )  ) |-> cfg_18 == rst_12 ; endproperty 
 
 property SyncReseteotid; @(negedge async_clk_12) ! (  auth_1  && hw_6 )  |-> core_1 == rst_20 ; endproperty 
 
 property SafeSendeotid; @(negedge async_clk_12) ! (  auth_1  && hw_6 ) && (  core_1  || sig_6  || hw_16 ) |-> clk_20 == tx_18; endproperty 
 
 property SecureSynceotid; @(negedge async_clk_12) ! (  auth_1  && hw_6 )  &&  ( !(  core_1  || sig_6  || hw_16 ) && (  auth_20  || core_20

In [19]:
wandb.finish()

eval/entropy,█▄▅▅▃▂▂▂▂▁▁
eval/loss,█▅▅▄▄▃▂▂▁▁▁
eval/mean_token_accuracy,▁▃▃▄▄▅▅▆▇██
eval/num_tokens,▁▂▂▃▄▅▅▆▇▇█
eval/runtime,▆▂▆▅▄▃▆▁▅█▂
eval/samples_per_second,▃▆▃▃▅▆▃█▃▁▆
eval/steps_per_second,▅█▅▅▅█▅█▅▁█
train/entropy,█▅▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/epoch,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇▇████
train/global_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇█
+5,...


## 9. Batch Inference on Test Set (vLLM)

For full test set inference, use vLLM for speed. Run this after training all 3 adapters.

In [ ]:
# !pip install -q vllm

In [ ]:
# ============================================================
# vLLM batch inference — uncomment and run after training
# ============================================================

# from vllm import LLM, SamplingParams
# from vllm.lora.request import LoRARequest
# import json
#
# ADAPTER_PATH = "/content/drive/MyDrive/sva_qlora/sva-qlora-verified/final_adapter"
# ADAPTER_NAME = "verified"  # change per adapter
#
# # Load base model with LoRA support
# llm = LLM(
#     model=BASE_MODEL,
#     enable_lora=True,
#     max_lora_rank=LORA_RANK,
#     max_model_len=MAX_SEQ_LENGTH,
#     trust_remote_code=True,
#     dtype="bfloat16",
# )
#
# sampling_params = SamplingParams(
#     temperature=0.1,
#     top_p=0.95,
#     max_tokens=2048,
# )
#
# # Load test set
# test_dataset = load_dataset(DATASET_REPO, DATASET_CONFIG, split="test")
#
# # Build prompts
# prompts = []
# for example in test_dataset:
#     messages = [
#         {"role": "system", "content": SYSTEM_PROMPT},
#         {"role": "user", "content": make_user_prompt(example["rtl"])},
#     ]
#     text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
#     prompts.append(text)
#
# # Generate with LoRA adapter
# lora_request = LoRARequest(ADAPTER_NAME, 1, ADAPTER_PATH)
# outputs = llm.generate(prompts, sampling_params, lora_request=lora_request)
#
# # Save results
# results = []
# for i, output in enumerate(outputs):
#     results.append({
#         "id": test_dataset[i].get("id", str(i)),
#         "rtl": test_dataset[i]["rtl"],
#         "generated_sva": output.outputs[0].text,
#         "reference_sva": test_dataset[i]["sva"],
#         "adapter": ADAPTER_NAME,
#     })
#
# output_file = f"/content/drive/MyDrive/sva_qlora/inference_{ADAPTER_NAME}.json"
# with open(output_file, "w") as f:
#     json.dump(results, f, indent=2)
# print(f"Saved {len(results)} results to {output_file}")

## 10. Training Summary

After completing all 3 runs, you should have:
```
/content/drive/MyDrive/sva_qlora/
├── sva-qlora-all/final_adapter/
├── sva-qlora-syntax_pass/final_adapter/
├── sva-qlora-verified/final_adapter/
├── inference_all.json
├── inference_syntax_pass.json
└── inference_verified.json
```

Feed each `inference_*.json` through JasperGold to get pass rates, then compare with ChatGPT baseline.